In [3]:
import sys; sys.path.insert(0, "build")
import numpy as np, pandas as pd, scipy.sparse as sp, glob, os
from sceris.reference import load_basis, apply_basis
from sceris.signatures import Signatures
from build.compartments import load_graph, build_mapper
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

BASES = "/home/vlad/scEris/data/universal_bases"
R = "/home/vlad/scRNA-profile-embedder-for-cross-dataset-retrieval/data/crc"


In [4]:
counts = sp.load_npz(f"{R}/counts_hv.npz")
genes = pd.read_csv(f"{R}/hv_genes.csv")["gene"].astype(str).tolist()
cells = pd.read_parquet(f"{R}/cells.parquet").reset_index(drop = True)
cells["cell_type"] = pd.read_parquet(f"{R}/cell_types.parquet")["cell_type"].values
cells["label"] = (cells.sample_type == "tumor").astype(int)
cells 

,sample_id,sample_type,study,assay,donor,cell_type,label
0,Borras_2023_KUL5_CD45Pos.EXT097,tumor,Borras_2023_Cell_Discov,10x 5' transcription profiling,Joanito_2022_Nat_Genet.SC040,"CD8-positive, alpha-beta T cell",1
1,Borras_2023_KUL5_CD45Pos.EXT097,tumor,Borras_2023_Cell_Discov,10x 5' transcription profiling,Joanito_2022_Nat_Genet.SC040,"CD4-positive, alpha-beta T cell",1
2,Borras_2023_KUL5_CD45Pos.EXT097,tumor,Borras_2023_Cell_Discov,10x 5' transcription profiling,Joanito_2022_Nat_Genet.SC040,"CD4-positive, alpha-beta T cell",1
3,Borras_2023_KUL5_CD45Pos.EXT097,tumor,Borras_2023_Cell_Discov,10x 5' transcription profiling,Joanito_2022_Nat_Genet.SC040,"CD4-positive, alpha-beta T cell",1
4,Borras_2023_KUL5_CD45Pos.EXT097,tumor,Borras_2023_Cell_Discov,10x 5' transcription profiling,Joanito_2022_Nat_Genet.SC040,regulatory T cell,1
...,...,...,...,...,...,...,...
317309,deVries_2023_LUMC.HTO8,tumor,deVries_2023_Nature,10x 5' v1,deVries_2023_Nature.HTO8,"CD8-positive, alpha-beta T cell",1
317310,deVries_2023_LUMC.HTO7,tumor,deVries_2023_Nature,10x 5' v1,deVries_2023_Nature.HTO7,"CD8-positive, alpha-beta T cell",1
317311,deVries_2023_LUMC.HTO9,tumor,deVries_2023_Nature,10x 5' v1,deVries_2023_Nature.HTO9,"CD8-positive, alpha-beta T cell",1
317312,deVries_2023_LUMC.HTO7,tumor,deVries_2023_Nature,10x 5' v1,deVries_2023_Nature.HTO7,gamma-delta T cell,1


In [5]:
mapper = build_mapper(load_graph())
cells["compartment"] = cells.cell_type.map({c: mapper(c, None) for c in cells.cell_type.unique()})
cells.groupby("compartment")
#print(cells["compartment"].unique())
cells.groupby("compartment").agg(
    n_cells=("cell_type", "size"),
    n_types=("cell_type", "nunique"),
    n_patients=("sample_id", "nunique"),
).sort_values("n_cells", ascending=False)


,n_cells,n_types,n_patients
compartment,,,
T_cell,116662,5,612
B_plasma,70489,8,622
neoplastic,40815,1,340
epithelial,29732,7,284
myeloid,22042,5,525
fibroblast,14565,1,455
endothelial,8378,3,436
pericyte,4761,1,410
mast,3109,1,401


In [12]:
def within_study_auc(X, y, studies, k = 3):      # X уже пациент-уровневый
    aucs = []
    for s in np.unique(studies):
        m = studies == s      # маска для пациентов из этой студии.
        vals, cnts = np.unique(y[m], return_counts=True)
        if len(vals) < 2 or cnts.min() < k:    
            continue
        Xs, ys = StandardScaler().fit_transform(X[m]), y[m]  # z преобразование
        p = np.zeros(len(ys))
        for tr, te in StratifiedKFold(k, shuffle = True, random_state = 0).split(Xs, ys):
            p[te] = LogisticRegression(C=0.1, max_iter = 2000).fit(Xs[tr], ys[tr]).predict_proba(Xs[te])[:,1]   # берем столбец вероятностей позитивного класса (болезнь)
        aucs.append(roc_auc_score(ys, p))
    return float(np.mean(aucs)) if aucs else np.nan

def batch_ratio_fixed_bio(X, studies, y, label = 1):
    m = y == label  
    Xs, st = X[m], studies[m]
    keep = [s for s in np.unique(st) if (st == s).sum() >= 3]  # отбираем достаточно большие студии
    if len(keep) < 2:  # что бы было что мерять впринципе
        return np.nan
    cents = {s: Xs[st == s].mean(0) for s in keep}  # центроиды по студиям
    cross = np.mean([np.linalg.norm(cents[a] - cents[b]) for i, a in enumerate(keep) for b in keep[i+1:]]) 
    within = np.mean([np.linalg.norm(Xs[st == s] - cents[s], axis = 1).mean() for s in keep])
    return float(cross / within)      

In [13]:
rows = []
for path in sorted(glob.glob(f"{BASES}/*.npz")):
    comp = os.path.splitext(os.path.basename(path))[0]
    m = (cells.compartment == comp).values
    if m.sum() < 500:
        continue
    coords = apply_basis(load_basis(path), counts[m], genes)         
    sub = cells[m]
    sig = Signatures(rff_dim=1024).fit(coords).patient(coords, sub.sample_id.values)
    pid = sig.index.to_numpy()
    X = sig.to_numpy()
    meta = sub.groupby("sample_id").agg(label=("label", "first"), study=("study", "first"))
    y = meta.loc[pid, "label"].to_numpy()
    studies = meta.loc[pid, "study"].to_numpy()
    rows.append(dict(compartment=comp, n_cells=int(m.sum()), n_pat=len(pid),
                     ceiling=within_study_auc(X, y, studies),
                     batch_ratio=batch_ratio_fixed_bio(X, studies, y)))

res = pd.DataFrame(rows).sort_values("ceiling", ascending=False)
print(res.to_string(index=False))


compartment  n_cells  n_pat  ceiling  batch_ratio
      other    45949    507 0.998036     1.535844
 epithelial    29732    284 0.995536     1.072133
endothelial     8378    436 0.931573     1.421798
 fibroblast    14565    455 0.898748     1.348778
     T_cell   116662    612 0.897882     2.066610
   B_plasma    70489    622 0.876833     1.510598
    myeloid    25151    555 0.836444     1.526565
       glia     1319    227 0.809336     1.140172
  dendritic     2892    478 0.743884     1.213350
    NK_cell     2177    464 0.723595     1.334164


In [1]:
def _boot_ci(vals, B=2000, seed=0):
    vals = np.asarray(vals, float)
    if len(vals) == 0:
        return (np.nan, np.nan, np.nan)
    if len(vals) < 2:
        return (float(vals.mean()), np.nan, np.nan)
    rng = np.random.default_rng(seed)
    bs = np.array([rng.choice(vals, len(vals), replace=True).mean() for _ in range(B)])
    return float(vals.mean()), float(np.percentile(bs, 2.5)), float(np.percentile(bs, 97.5))


def within_study_auc(X, y, studies, k=3):
    aucs = []
    for s in np.unique(studies):
        m = studies == s
        vals, cnts = np.unique(y[m], return_counts=True)
        if len(vals) < 2 or cnts.min() < k:
            continue
        Xs, ys = StandardScaler().fit_transform(X[m]), y[m]
        p = np.zeros(len(ys))
        for tr, te in StratifiedKFold(k, shuffle=True, random_state=0).split(Xs, ys):
            p[te] = LogisticRegression(C=0.1, max_iter=2000).fit(Xs[tr], ys[tr]).predict_proba(Xs[te])[:, 1]
        aucs.append(roc_auc_score(ys, p))
    return _boot_ci(aucs)                      # (mean, lo, hi) по студиям


def study_divergence_same_dx(X, studies, y, label=1):
    m = y == label
    Xs, st = X[m], studies[m]
    keep = [s for s in np.unique(st) if (st == s).sum() >= 3]
    if len(keep) < 2:
        return (np.nan, np.nan, np.nan)
    cents = {s: Xs[st == s].mean(0) for s in keep}
    g = np.mean([cents[s] for s in keep], 0)
    b = [np.linalg.norm(cents[s] - g) / np.linalg.norm(Xs[st == s] - cents[s], axis=1).mean() for s in keep]
    return _boot_ci(b)                         # (mean, lo, hi) по студиям


In [6]:
rows = []
for path in sorted(glob.glob(f"{BASES}/*.npz")):
    comp = os.path.splitext(os.path.basename(path))[0]
    m = (cells.compartment == comp).values
    if m.sum() < 500:
        continue
    coords = apply_basis(load_basis(path), counts[m], genes)
    sub = cells[m]
    sig = Signatures(rff_dim=1024).fit(coords).patient(coords, sub.sample_id.values)
    pid = sig.index.to_numpy(); X = sig.to_numpy()
    meta = sub.groupby("sample_id").agg(label=("label", "first"), study=("study", "first"))
    y = meta.loc[pid, "label"].to_numpy(); studies = meta.loc[pid, "study"].to_numpy()
    c = within_study_auc(X, y, studies)
    b = study_divergence_same_dx(X, studies, y)
    rows.append(dict(compartment=comp, n_pat=len(pid),
                     ceiling=c[0], c_lo=c[1], c_hi=c[2],
                     batch=b[0], b_lo=b[1], b_hi=b[2]))

res = pd.DataFrame(rows).sort_values("ceiling", ascending=False)
for _, r in res.iterrows():
    print(f"{r.compartment:12s} n={int(r.n_pat):4d}  "
          f"ceiling {r.ceiling:.3f} [{r.c_lo:.3f}, {r.c_hi:.3f}]   "
          f"batch {r.batch:.2f} [{r.b_lo:.2f}, {r.b_hi:.2f}]")


epithelial   n= 284  ceiling 0.996 [0.987, 1.000]   batch 0.75 [0.57, 0.97]
endothelial  n= 436  ceiling 0.932 [0.896, 0.963]   batch 1.02 [0.81, 1.24]
fibroblast   n= 455  ceiling 0.899 [0.814, 0.962]   batch 0.97 [0.78, 1.18]
T_cell       n= 612  ceiling 0.898 [0.836, 0.953]   batch 1.52 [1.25, 1.77]
B_plasma     n= 622  ceiling 0.877 [0.803, 0.942]   batch 1.11 [0.94, 1.29]
myeloid      n= 525  ceiling 0.836 [0.776, 0.899]   batch 1.08 [0.88, 1.31]
glia         n= 227  ceiling 0.809 [0.734, 0.871]   batch 0.79 [0.59, 0.99]
dendritic    n= 478  ceiling 0.744 [0.632, 0.846]   batch 0.86 [0.74, 0.99]
NK_cell      n= 464  ceiling 0.724 [0.594, 0.843]   batch 0.98 [0.79, 1.20]
